# Week 6 -- Function 7: GP + UCB Exploitation (6D)
**ML Hyperparameter Tuning** — Goldilocks type. W3 sweet spot at [0.074, 0.561, 0.359, 0.097, 0.306, 0.674] scored 1.236.

In [1]:
# -- WEEKLY OBSERVATIONS LOG
import numpy as np

INITIAL_SIZE = 30
DATA_PATH_X  = '../data/function_7/initial_inputs.npy'
DATA_PATH_Y  = '../data/function_7/initial_outputs.npy'

# Key insights:
# - x1 should be ~0.07, NOT zero (W4 x1=0, W5 x1=0.014 both worse than W3 x1=0.074)
# - x6 near 0.67 is critical -- W1 x6=0.88 and W4 x6=0.88 both scored much worse
weekly_log = [
    ([0.060571, 0.494947, 0.127446, 0.141829, 0.443814, 0.875291], 0.679156918097955),   # W1: x6=0.875 too high
    ([0.071207, 0.491965, 0.232822, 0.256730, 0.537272, 0.628233], 0.7752482092923055),  # W2: improving
    ([0.074437, 0.560683, 0.358665, 0.096698, 0.306449, 0.674354], 1.2355031217908),     # W3: BEST
    ([0.000000, 0.350811, 0.312286, 0.068118, 0.270428, 0.880970], 0.997758497067847),   # W4: x1=0, x6=0.88 -- worse
    ([0.014437, 0.567744, 0.308649, 0.134874, 0.366449, 0.698014], 1.1313122321728861),  # W5: x1 too low -- worse
]

X_disk = np.load(DATA_PATH_X)
Y_disk = np.load(DATA_PATH_Y)

n_missing = (INITIAL_SIZE + len(weekly_log)) - len(Y_disk)
if n_missing > 0:
    new_entries = weekly_log[len(weekly_log) - n_missing:]
    for x_vals, y_val in new_entries:
        X_disk = np.vstack([X_disk, np.array([x_vals])])
        Y_disk = np.append(Y_disk, y_val)
    np.save(DATA_PATH_X, X_disk)
    np.save(DATA_PATH_Y, Y_disk)
    print(f'Synced {n_missing} new observation(s).')
else:
    print('Already up-to-date.')

print(f'X: {X_disk.shape} | Y: {Y_disk.shape}')
current_week = len(Y_disk) - INITIAL_SIZE + 1
print(f'Week {current_week} | {len(Y_disk)} total observations ({INITIAL_SIZE} initial + {len(Y_disk)-INITIAL_SIZE} added)')

Already up-to-date.
X: (35, 6) | Y: (35,)
Week 6 | 35 total observations (30 initial + 5 added)


In [2]:
# Cell 3: Load data and inspect -- Function 7: ML Hyperparameter Tuning (6D), Maximise

X = np.load(DATA_PATH_X)
Y = np.load(DATA_PATH_Y)
n_dims = X.shape[1]

print(f'Input  shape : {X.shape}')
print(f'Output shape : {Y.shape}')
print()

# Sort by Y descending -- higher is better
pairs = sorted(zip(Y.tolist(), X.tolist()), reverse=True)
Y_sorted = [p[0] for p in pairs]
X_sorted = [p[1] for p in pairs]

print('=' * 90)
print('  All observations (sorted descending by Y)   x1 and x6 are the key dimensions')
print('=' * 90)
for i, (y_val, x_val) in enumerate(zip(Y_sorted, X_sorted)):
    marker = '  <-- best' if i == 0 else ''
    x_str = ', '.join(f'{v:.4f}' for v in x_val)
    print(f'  [{i+1:2d}]  X=[{x_str}]  Y={y_val:+.4f}  x1={x_val[0]:.3f}  x6={x_val[5]:.3f}{marker}')
print('=' * 90)

best_Y = Y_sorted[0]
best_X = np.array(X_sorted[0])
best_x_str = ', '.join(f'{v:.6f}' for v in best_X)
print(f'\n  Best Y*  = {best_Y:.4f}')
print(f'  Best X*  = [{best_x_str}]')
print(f'  x1*={best_X[0]:.4f}  x6*={best_X[5]:.4f}')

Input  shape : (35, 6)
Output shape : (35,)

  All observations (sorted descending by Y)   x1 and x6 are the key dimensions
  [ 1]  X=[0.0579, 0.4917, 0.2474, 0.2181, 0.4204, 0.7310]  Y=+1.3650  x1=0.058  x6=0.731  <-- best
  [ 2]  X=[0.0744, 0.5607, 0.3587, 0.0967, 0.3064, 0.6744]  Y=+1.2355  x1=0.074  x6=0.674
  [ 3]  X=[0.0144, 0.5677, 0.3086, 0.1349, 0.3664, 0.6980]  Y=+1.1313  x1=0.014  x6=0.698
  [ 4]  X=[0.0000, 0.3508, 0.3123, 0.0681, 0.2704, 0.8810]  Y=+0.9978  x1=0.000  x6=0.881
  [ 5]  X=[0.0712, 0.4920, 0.2328, 0.2567, 0.5373, 0.6282]  Y=+0.7752  x1=0.071  x6=0.628
  [ 6]  X=[0.0606, 0.4949, 0.1274, 0.1418, 0.4438, 0.8753]  Y=+0.6792  x1=0.061  x6=0.875
  [ 7]  X=[0.8816, 0.2045, 0.4145, 0.4204, 0.2649, 0.7307]  Y=+0.6751  x1=0.882  x6=0.731
  [ 8]  X=[0.1486, 0.0339, 0.7288, 0.3161, 0.0218, 0.5169]  Y=+0.6115  x1=0.149  x6=0.517
  [ 9]  X=[0.2726, 0.3245, 0.8971, 0.8330, 0.1541, 0.7959]  Y=+0.6044  x1=0.273  x6=0.796
  [10]  X=[0.5430, 0.9247, 0.3416, 0.6465, 0.7184, 0.343

In [3]:
# Cell 4: Fit GP
# F7 values range 0.4 to 1.3 -- tight normal range, raw Y fit.
# length_scale=0.3: in 6D with ~35 points, 0.1 leaves data islands.
# alpha=1e-4 for noise tolerance.
import warnings
warnings.filterwarnings('ignore')
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF

kernel = RBF(length_scale=0.3, length_scale_bounds='fixed')
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-4)
gp.fit(X, Y)

w3_anchor = np.array([0.074437, 0.560683, 0.358665, 0.096698, 0.306449, 0.674354])

print('=' * 62)
print('  GP Fitting Results')
print('=' * 62)
print(f'  Kernel                  : {gp.kernel_}')
print(f'  Alpha (noise)           : 1e-4')
print(f'  Log-marginal-likelihood : {gp.log_marginal_likelihood_value_:.4f}')
pred_mean, pred_std = gp.predict(best_X.reshape(1, -1), return_std=True)
print('  Sanity check at best known X* (W3):')
print(f'    GP predicted mean      = {pred_mean[0]:.4f}')
print(f'    Actual Y* (W3)         = {best_Y:.4f}')
print(f'    GP predicted std       = {pred_std[0]:.6f}')
pred_w4, _ = gp.predict(np.array([[0.0, 0.350811, 0.312286, 0.068118, 0.270428, 0.880970]]),
                         return_std=True)
print(f'  GP at W4 [x1=0, x6=0.88] : predicted={pred_w4[0]:.4f}  (actual: 0.9978)')
print('=' * 62)

  GP Fitting Results
  Kernel                  : RBF(length_scale=0.3)
  Alpha (noise)           : 1e-4
  Log-marginal-likelihood : -29.5280
  Sanity check at best known X* (W3):
    GP predicted mean      = 1.3642
    Actual Y* (W3)         = 1.3650
    GP predicted std       = 0.009989
  GP at W4 [x1=0, x6=0.88] : predicted=0.9978  (actual: 0.9978)


In [4]:
# Cell 5: GP UCB Acquisition -- focused random search around W3 sweet spot
# Radius ±0.08: keeps x1 within [0.074-0.08, 0.074+0.08] = [-0.006, 0.154] -> clipped to [0, 0.154]
# Tighter than ±0.12 to prevent x1 drifting above 0.15 or x6 above 0.80.

w3_best = np.array([0.074437, 0.560683, 0.358665, 0.096698, 0.306449, 0.674354])
np.random.seed(42)
X_grid = np.clip(
    w3_best + np.random.uniform(-0.08, 0.08, size=(50000, 6)),
    0.0, 1.0
)

gp_mean, gp_std = gp.predict(X_grid, return_std=True)

# beta=1.0: exploit the W3 region
beta = 1.0
ucb_scores = gp_mean + beta * gp_std

best_idx = np.argmax(ucb_scores)
next_x = X_grid[best_idx]
portal_string = '-'.join([f'{v:.6f}' for v in next_x])

print('=' * 72)
print('  GP UCB Acquisition (beta=1.0, exploitation)')
print('=' * 72)
print(f'  Search : 50,000 random points within ±0.08 of W3 anchor')
print(f'  W3 anchor : [{" ".join(f"{v:.4f}" for v in w3_best)}]')
print(f'  x1 range  : [{max(0, w3_best[0]-0.08):.4f}, {w3_best[0]+0.08:.4f}]  (clipped to [0,1])')
print(f'  x6 range  : [{w3_best[5]-0.08:.4f}, {w3_best[5]+0.08:.4f}]')
print(f'  Beta : {beta}')
next_str = ', '.join(f'{v:.6f}' for v in next_x)
print(f'  Next query   : [{next_str}]')
print(f'  x1={next_x[0]:.4f}  x6={next_x[5]:.4f}')
print(f'  UCB score    : {ucb_scores[best_idx]:.4f}')
print(f'  GP mean      : {gp_mean[best_idx]:.4f}')
print(f'  GP std       : {gp_std[best_idx]:.6f}')
print('=' * 72)

  GP UCB Acquisition (beta=1.0, exploitation)
  Search : 50,000 random points within ±0.08 of W3 anchor
  W3 anchor : [0.0744 0.5607 0.3587 0.0967 0.3064 0.6744]
  x1 range  : [0.0000, 0.1544]  (clipped to [0,1])
  x6 range  : [0.5944, 0.7544]
  Beta : 1.0
  Next query   : [0.149588, 0.483875, 0.411552, 0.175677, 0.296274, 0.745243]
  x1=0.1496  x6=0.7452
  UCB score    : 1.9260
  GP mean      : 1.5895
  GP std       : 0.336547


In [5]:
# Cell 6: Sanity check

w3_anchor    = np.array([0.074437, 0.560683, 0.358665, 0.096698, 0.306449, 0.674354])
dist_to_w3   = np.linalg.norm(next_x - w3_anchor)
dist_to_best = np.linalg.norm(next_x - best_X)

x1_safe = 0.03 <= next_x[0] <= 0.15   # low but NOT zero
x6_safe = 0.55 <= next_x[5] <= 0.80   # mid -- high x6 kills performance

print('=' * 72)
print('  Sanity Check')
print('=' * 72)
next_str = ', '.join(f'{v:.6f}' for v in next_x)
w3_str   = ', '.join(f'{v:.6f}' for v in w3_anchor)
print(f'  Proposed query  : [{next_str}]')
print(f'  W3 anchor       : [{w3_str}]  (Y={best_Y:.4f})')
print(f'  Distance to W3  : {dist_to_w3:.6f}')
print(f'  x1={next_x[0]:.4f}  in [0.03, 0.15]: {x1_safe}  (low but not zero)')
print(f'  x6={next_x[5]:.4f}  in [0.55, 0.80]: {x6_safe}  (mid, not high)')
print()

if dist_to_w3 > 0.20:
    print('  WARNING: query is far from W3 sweet spot (> 0.20)!')
else:
    print('  OK: query is close to W3 sweet spot.')

if not x1_safe:
    print(f'  WARNING: x1={next_x[0]:.4f} outside [0.03,0.15] -- W4 (x1=0) and W5 (x1=0.014) both scored worse than W3!')
else:
    print(f'  OK: x1 is in the sweet range [0.03, 0.15].')

if not x6_safe:
    print(f'  WARNING: x6={next_x[5]:.4f} outside [0.55,0.80] -- W1 (x6=0.875) and W4 (x6=0.881) both scored much worse!')
else:
    print(f'  OK: x6 is in the safe mid-range [0.55, 0.80].')
print('=' * 72)

  Sanity Check
  Proposed query  : [0.149588, 0.483875, 0.411552, 0.175677, 0.296274, 0.745243]
  W3 anchor       : [0.074437, 0.560683, 0.358665, 0.096698, 0.306449, 0.674354]  (Y=1.3650)
  Distance to W3  : 0.160345
  x1=0.1496  in [0.03, 0.15]: True  (low but not zero)
  x6=0.7452  in [0.55, 0.80]: True  (mid, not high)

  OK: query is close to W3 sweet spot.
  OK: x1 is in the sweet range [0.03, 0.15].
  OK: x6 is in the safe mid-range [0.55, 0.80].


In [6]:
print('=' * 72)
print('  SUMMARY -- Week 6 Bayesian Optimisation')
print('=' * 72)
print('  Function        : 7 -- ML Hyperparameter Tuning (6D)')
print('  Objective       : Maximise Y')
print(f'  Method          : GP UCB (beta=1.0, random search ±0.12 of W3, raw Y, ls=0.3)')
print(f'  Key insight     : x1~0.07 (not zero), x6~0.67 (not high) -- Goldilocks')
print()
best_x_s = ', '.join(f'{v:.6f}' for v in best_X)
next_s   = ', '.join(f'{v:.6f}' for v in next_x)
print(f'  Best X* (W3)    : [{best_x_s}]')
print(f'  Best Y*         : {best_Y:.4f}')
print()
print(f'  Next query      : [{next_s}]')
print()
print('  Portal submission string:')
print(f'  >>> {portal_string} <<<')
print('=' * 72)

  SUMMARY -- Week 6 Bayesian Optimisation
  Function        : 7 -- ML Hyperparameter Tuning (6D)
  Objective       : Maximise Y
  Method          : GP UCB (beta=1.0, random search ±0.12 of W3, raw Y, ls=0.3)
  Key insight     : x1~0.07 (not zero), x6~0.67 (not high) -- Goldilocks

  Best X* (W3)    : [0.057896, 0.491672, 0.247422, 0.218118, 0.420428, 0.730970]
  Best Y*         : 1.3650

  Next query      : [0.149588, 0.483875, 0.411552, 0.175677, 0.296274, 0.745243]

  Portal submission string:
  >>> 0.149588-0.483875-0.411552-0.175677-0.296274-0.745243 <<<


## Week 6 -- Reflection

**Function**: F7 -- ML Hyperparameter Tuning (6D)

### Strategy change from Week 5
- Removed NN+SVM+gradient sensitivity: 4000+ params on 35 points in 6D overfits noise.
- Raw Y fit, length_scale=0.3: GP connects sparse 6D data.
- Random search ±0.12 around W3: avoids expensive structured grid in 6D.
- Hard constraints: x1∈[0.03,0.15], x6∈[0.55,0.80] — evidence shows outside these = worse.

### Key pattern
| Week | x1 | x6 | Y |
|------|-----|-----|------|
| W3 | 0.074 | 0.674 | **1.236** |
| W5 | 0.014 | 0.698 | 1.131 |
| W4 | 0.000 | 0.881 | 0.998 |
| W2 | 0.071 | 0.628 | 0.775 |
| W1 | 0.061 | 0.875 | 0.679 |

### Query
- **Input submitted**: *(fill in portal submission string)*
- **Output received**: *(fill in after result)*
- **Best so far**: *(update after result)*

### What this result taught us
*(fill in after receiving result)*

### Strategy for Week 7
*(fill in after reflecting on result)*